# Exercise 1.2 — Limited Angle Tomography


This notebook simulates noisy sinograms at three angular ranges: 180°, 120°, 40° 
reconstructs them with FBP and gradient descent, and analyses the effect of
angular coverage on image quality.

## 1. Imports and setup

In [ ]:
import os
import sys

sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage.transform import radon, iradon
from skimage.metrics import structural_similarity

from ct_reconstruction.phantom import load_ct_image

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.titlesize"] = 9

SAVE_DIR = "report_figures/exercise_1_2"
os.makedirs(SAVE_DIR, exist_ok=True)

## 2. Load phantom and apply circular mask

In [2]:
def circular_mask(shape):
    h, w = shape
    cy, cx = h / 2.0, w / 2.0
    yy, xx = np.ogrid[:h, :w]
    r = min(h, w) / 2.0
    return (yy - cy) ** 2 + (xx - cx) ** 2 <= r ** 2


def apply_circular_mask(image):
    mask = circular_mask(image.shape)
    out = image.copy().astype(np.float64)
    out[~mask] = 0.0
    return out, mask


ct_path = "../CT_exercise_1.png"
phantom = load_ct_image(ct_path).astype(np.float64)
phantom, recon_mask = apply_circular_mask(phantom)

print(f"Phantom shape : {phantom.shape}")
print(f"Pixel range   : [{phantom.min():.3f}, {phantom.max():.3f}]")

plt.figure(figsize=(4, 4))
plt.imshow(phantom, cmap="gray")
plt.title("CT phantom (masked to circle)")
plt.axis("off")
plt.tight_layout()
plt.show()

Phantom shape : (512, 512)
Pixel range   : [0.000, 1.000]


/var/folders/2f/7345nh9n0nlcj7ty8bmywp480000gn/T/ipykernel_28956/624344424.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Define angular ranges

Each angular range uses **360 projections** — the same view count as Exercise 1.1's
full-angle baseline — distributed uniformly over the scan arc.  Keeping the view count
fixed isolates the **angular span** effect from the **number-of-projections** effect:
only the range of directions measured changes, not how densely they are sampled.

Using 1 projection per degree (180/120/40 views) would confound both effects simultaneously,
making it impossible to attribute degradation to the missing wedge alone.

In [3]:
# 360 views for every angular range — isolates span from count.
# theta spacing differs across ranges; only the sampled arc changes.
N_VIEWS = 360

angle_configs = [
    (180, N_VIEWS),
    (120, N_VIEWS),
    ( 40, N_VIEWS),
]

for max_angle, n_proj in angle_configs:
    theta = np.linspace(0, max_angle, n_proj, endpoint=False)
    print(f"Range {max_angle:3d}deg  |  {n_proj} projections  |  "
          f"step {theta[1]-theta[0]:.3f}deg  |  "
          f"theta: {theta[0]:.1f} – {theta[-1]:.2f} deg")

Range 180deg  |  360 projections  |  step 0.500deg  |  theta: 0.0 – 179.50 deg
Range 120deg  |  360 projections  |  step 0.333deg  |  theta: 0.0 – 119.67 deg
Range  40deg  |  360 projections  |  step 0.111deg  |  theta: 0.0 – 39.89 deg


## 4. Combined noise model (Gaussian + Poisson)

Gaussian electronic detector noise and Poisson photon noise are applied together
in the transmission domain, following the physical acquisition order in Exercise 1.1
(Practical 2, Q3a–Q3b):

1. $p_\text{norm} = p / p_{\max}$ — normalise (practical hint: raw Radon values are too large)
2. $I = I_0 \exp(-p_\text{norm})$ — Beer-Lambert
3. $I_\text{Gauss} = I + \mathcal{N}(0, \sigma^2)$ — Gaussian detector noise added **first**
4. $I_\text{Poisson} \sim \text{Poisson}(\text{clip}(I_\text{Gauss}))$ — photon noise applied **second**
5. $p_\text{noisy} = -\log(I_\text{Poisson} / I_0) \cdot p_{\max}$ — back to attenuation scale

Gaussian is added before Poisson to match Exercise 1.1's `simulate_noisy_sinogram`.

In [4]:
def simulate_noisy_sinogram(sinogram, I0, sigma=0.05, rng=None):
    """
    Simulate a noisy sinogram — identical pipeline to Exercise 1.1.

    Pipeline (Practical 2, Q3a-Q3b):
      1. p_norm = sinogram / p_max
      2. I = I0 * exp(-p_norm)               -- Beer-Lambert
      3. I_gauss = I + N(0, sigma^2)         -- Gaussian detector noise first
      4. clip(I_gauss, 1e-8, None)
      5. I_noisy ~ Poisson(I_gauss)          -- Poisson photon noise second
      6. clip(I_noisy, 1, None)
      7. p_noisy = -log(I_noisy / I0) * p_max
    """
    if rng is None:
        rng = np.random.default_rng()

    p_max = sinogram.max()
    if p_max == 0:
        return sinogram.copy()

    p_norm  = sinogram / p_max
    I       = I0 * np.exp(-p_norm)
    I_gauss = I + rng.normal(0.0, sigma, size=sinogram.shape)   # Gaussian first
    I_gauss = np.clip(I_gauss, 1e-8, None)
    I_noisy = rng.poisson(I_gauss).astype(np.float64)           # Poisson second
    I_noisy = np.clip(I_noisy, 1, None)

    return -np.log(I_noisy / I0) * p_max

## 5. Build the experiment grid

3 angular ranges × 3 dose levels = 9 conditions.
Each sinogram has both Poisson and Gaussian noise applied in the transmission domain.

In [5]:
rng     = np.random.default_rng(42)
I0_list = [1e5, 1e3, 1e2]
sigma   = 0.05    # Gaussian detector noise std (in transmission counts)

noisy_results = {}

for max_angle, n_proj in angle_configs:
    noisy_results[max_angle] = {}
    theta      = np.linspace(0, max_angle, n_proj, endpoint=False)
    sino_clean = radon(phantom, theta=theta, circle=True)

    for I0 in I0_list:
        sino_noisy = simulate_noisy_sinogram(sino_clean, I0=I0, sigma=sigma, rng=rng)

        noisy_results[max_angle][I0] = {
            "theta":      theta,
            "sino_clean": sino_clean,
            "sino_noisy": sino_noisy,
            "I0":         I0,
        }

    print(f"{max_angle:3d}deg  sinogram {sino_clean.shape}  "
          f"range [{sino_clean.min():.2f}, {sino_clean.max():.2f}]")

180deg  sinogram (512, 360)  range [0.00, 101.31]


120deg  sinogram (512, 360)  range [0.00, 101.14]


 40deg  sinogram (512, 360)  range [0.00, 87.99]


## 6. Visualise noisy sinograms (Poisson + Gaussian)

In [ ]:
fig, axes = plt.subplots(len(I0_list), len(angle_configs), figsize=(15, 10))

for i, I0 in enumerate(I0_list):
    for j, (max_angle, _) in enumerate(angle_configs):
        ax      = axes[i, j]
        p_noisy = noisy_results[max_angle][I0]["sino_noisy"]

        im = ax.imshow(p_noisy, cmap="gray", aspect="auto")
        ax.set_title(f"Poisson + Gaussian noise\nI0={I0:.0e}, range={max_angle}deg")
        ax.set_xlabel("Projection index")
        ax.set_ylabel("Detector bin")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle(f"Limited-angle noisy sinograms (sigma={sigma})", fontsize=12)
plt.tight_layout()
fig.savefig(f"{SAVE_DIR}/noisy_sinograms.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Reconstruction functions

### FBP 
Ramp filter applied to the sinogram, followed by backprojection.

### Gradient Descent 

    x^{k+1} = x^k - gamma * A^T(Ax^k - b)

where A = radon(), A^T = iradon(..., filter_name=None), gamma = 0.001.

In [7]:
def reconstruct_fbp(sinogram, theta, output_size):
    """Filtered backprojection with ramp filter (Practical 2, Q5)."""
    return iradon(
        sinogram, theta=theta, filter_name="ramp",
        circle=True, output_size=output_size,
    )


def reconstruct_gd(sinogram, theta, output_size, phantom=None, mask=None,
                   gamma=0.001, n_iter=200):
    """
    Gradient descent reconstruction (Practical 2, Q6).

    Update rule:  x^{k+1} = x^k - gamma * A^T(Ax^k - b)
    where A = radon(), A^T = iradon(..., filter_name=None).

    Residual MSE and image MSE are logged at x^k (before the update).
    """
    x = np.zeros((output_size, output_size), dtype=np.float64)
    residual_losses  = []
    image_mse_losses = []

    for _ in range(n_iter):
        residual = radon(x, theta=theta, circle=True) - sinogram

        # Log at x^k before stepping
        residual_losses.append(float(np.mean(residual ** 2)))

        if phantom is not None:
            ref = phantom[mask] if mask is not None else phantom.ravel()
            est = x[mask]       if mask is not None else x.ravel()
            image_mse_losses.append(float(np.mean((ref - est) ** 2)))

        # A^T (Ax^k - b): unfiltered backprojection of the residual
        grad = iradon(residual, theta=theta, filter_name=None,
                      circle=True, output_size=output_size)
        x = x - gamma * grad

    return x, residual_losses, image_mse_losses

In [8]:
def compute_metrics(reference, reconstruction, mask):
    """Masked MSE, PSNR, SSIM evaluated inside the reconstruction circle."""
    ref = np.asarray(reference, dtype=np.float64)
    rec = np.asarray(reconstruction, dtype=np.float64)

    ref_vals   = ref[mask]
    rec_vals   = rec[mask]
    mse        = float(np.mean((ref_vals - rec_vals) ** 2))
    data_range = float(ref_vals.max() - ref_vals.min()) or 1.0
    psnr       = 10.0 * np.log10(data_range ** 2 / mse) if mse > 0 else np.inf

    ref2d = np.zeros_like(ref);  ref2d[mask] = ref[mask]
    rec2d = np.zeros_like(rec);  rec2d[mask] = rec[mask]
    ssim  = float(structural_similarity(ref2d, rec2d, data_range=data_range))

    return {"MSE": mse, "PSNR": psnr, "SSIM": ssim}

## 8. Reconstruct all experiments (FBP + GD)

In [9]:
output_size = phantom.shape[0]
recon_results = {}

for max_angle, _ in angle_configs:
    recon_results[max_angle] = {}

    for I0 in I0_list:
        case    = noisy_results[max_angle][I0]
        theta   = case["theta"]
        p_noisy = case["sino_noisy"]

        print(f"Reconstructing {max_angle}deg  I0={I0:.0e} ...", end="  ")

        fbp_noisy = reconstruct_fbp(p_noisy, theta, output_size)

        gd_noisy, gd_res, gd_img = reconstruct_gd(
            p_noisy, theta, output_size,
            phantom=phantom, mask=recon_mask,
        )

        recon_results[max_angle][I0] = {
            "fbp_noisy":                 fbp_noisy,
            "gd_noisy":                  gd_noisy,
            "gd_noisy_residual_losses":  gd_res,
            "gd_noisy_image_mse_losses": gd_img,
            "fbp_metrics_noisy": compute_metrics(phantom, fbp_noisy, recon_mask),
            "gd_metrics_noisy":  compute_metrics(phantom, gd_noisy,  recon_mask),
        }
        print("done")

print("All reconstructions complete.")

Reconstructing 180deg  I0=1e+05 ...  

done
Reconstructing 180deg  I0=1e+03 ...  

done
Reconstructing 180deg  I0=1e+02 ...  

done
Reconstructing 120deg  I0=1e+05 ...  

done
Reconstructing 120deg  I0=1e+03 ...  

done
Reconstructing 120deg  I0=1e+02 ...  

done
Reconstructing 40deg  I0=1e+05 ...  

done
Reconstructing 40deg  I0=1e+03 ...  

done
Reconstructing 40deg  I0=1e+02 ...  

done
All reconstructions complete.


## 9. Metrics helper

## 10. GD convergence plots

In [ ]:
ordered_I0     = sorted(I0_list, reverse=True)
ordered_angles = sorted([a for a, _ in angle_configs], reverse=True)

fig, axes = plt.subplots(
    len(ordered_I0), len(ordered_angles),
    figsize=(15, 10), sharex=True, squeeze=False,
)

for i, I0 in enumerate(ordered_I0):
    for j, max_angle in enumerate(ordered_angles):
        ax   = axes[i, j]
        case = recon_results[max_angle][I0]
        sino = noisy_results[max_angle][I0]["sino_noisy"]

        rel_res = (np.sqrt(np.asarray(case["gd_noisy_residual_losses"]) * sino.size)
                   / (np.linalg.norm(sino) + 1e-12))
        img_mse = np.asarray(case["gd_noisy_image_mse_losses"])

        ax.plot(rel_res, linewidth=2, label="Relative residual")
        ax.plot(img_mse, linewidth=2, label="Image MSE (masked)")
        ax.set_title(f"I0={I0:.0e}, range={max_angle}°", fontsize=10)
        ax.set_xlabel("Iteration")
        if j == 0:
            ax.set_ylabel("Loss")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False,
           bbox_to_anchor=(0.5, 1.02))
fig.suptitle("GD convergence — limited angle, Poisson + Gaussian noise", fontsize=13)
plt.tight_layout(rect=(0, 0, 1, 0.96))
fig.savefig(f"{SAVE_DIR}/gd_convergence.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Reconstruction visual comparison by dose level

In [ ]:
def _masked_view(image, mask):
    return np.ma.masked_where(~mask, image)


def _compute_shared_visual_scales(recon_results, phantom, recon_mask, angle_configs, I0_list):
    image_values = [phantom[recon_mask].ravel()]
    error_values = []
    for max_angle, _ in angle_configs:
        for I0 in I0_list:
            case = recon_results[max_angle][I0]
            for key in ("fbp_noisy", "gd_noisy"):
                r = case[key]
                image_values.append(r[recon_mask].ravel())
                error_values.append(np.abs(phantom - r)[recon_mask].ravel())
    img_vmin, img_vmax = np.percentile(np.concatenate(image_values), [1, 99])
    err_vmax = max(np.percentile(np.concatenate(error_values), 99.5), 1e-12)
    return {"image_vmin": img_vmin, "image_vmax": img_vmax, "error_vmax": err_vmax}


def _plot_reconstruction_report(rows, phantom, recon_mask, scales, figure_title, save_path=None):
    gray_cm = plt.get_cmap("gray").copy();    gray_cm.set_bad("white")
    err_cm  = plt.get_cmap("inferno").copy(); err_cm.set_bad("white")
    col_titles = ["FBP", "FBP error", "GD", "GD error"]

    fig, axes = plt.subplots(len(rows), 4, figsize=(12, 2.8 * len(rows)),
                             constrained_layout=True)
    if len(rows) == 1:
        axes = axes[None, :]

    err_im = None
    for ri, row in enumerate(rows):
        fbp, gd = row["fbp"], row["gd"]
        panels = [
            (_masked_view(fbp,                  recon_mask), gray_cm, scales["image_vmin"], scales["image_vmax"]),
            (_masked_view(np.abs(phantom - fbp), recon_mask), err_cm, 0, scales["error_vmax"]),
            (_masked_view(gd,                   recon_mask), gray_cm, scales["image_vmin"], scales["image_vmax"]),
            (_masked_view(np.abs(phantom - gd),  recon_mask), err_cm, 0, scales["error_vmax"]),
        ]
        for ci, (img, cmap, vmin, vmax) in enumerate(panels):
            ax = axes[ri, ci]
            im = ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax, interpolation="none")
            ax.set_xticks([]); ax.set_yticks([])
            ax.set_facecolor("white")
            for sp in ax.spines.values(): sp.set_visible(False)
            if ri == 0:
                ax.set_title(col_titles[ci], fontsize=11, fontweight="semibold")
            if ci in (1, 3):
                err_im = im
        axes[ri, 0].set_ylabel(row["label"], fontsize=10, fontweight="semibold")

    fig.colorbar(err_im, ax=axes[:, 3], shrink=0.9, label="Absolute error")
    fig.suptitle(figure_title, fontsize=13, fontweight="semibold")
    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


visual_scales = _compute_shared_visual_scales(
    recon_results, phantom, recon_mask, angle_configs, I0_list,
)

I0_label_map = {1e5: "1e5", 1e3: "1e3", 1e2: "1e2"}

for I0 in sorted(I0_list, reverse=True):
    rows = []
    for max_angle, _ in sorted(angle_configs, reverse=True):
        case = recon_results[max_angle][I0]
        rows.append({"label": f"{max_angle}°", "fbp": case["fbp_noisy"], "gd": case["gd_noisy"]})
    _plot_reconstruction_report(
        rows, phantom, recon_mask, visual_scales,
        figure_title=f"Limited angle — Poisson + Gaussian noise (I0={I0:.0e}, sigma={sigma})",
        save_path=f"{SAVE_DIR}/recon_I0_{I0_label_map[I0]}.png",
    )

## 12. Quantitative metrics table

In [12]:
rows = []

for max_angle, _ in angle_configs:
    for I0 in I0_list:
        fbp_m = recon_results[max_angle][I0]["fbp_metrics_noisy"]
        gd_m  = recon_results[max_angle][I0]["gd_metrics_noisy"]

        for method, m in [("FBP", fbp_m), ("GD", gd_m)]:
            rows.append({
                "Angular range (deg)": max_angle,   # int for correct numeric sort
                "I0":     I0,
                "Method": method,
                "MSE":    round(m["MSE"],  6),
                "PSNR":   round(m["PSNR"], 2),
                "SSIM":   round(m["SSIM"], 4),
            })

metrics_df = pd.DataFrame(rows).sort_values(
    ["I0", "Angular range (deg)", "Method"], ascending=[False, False, True]
).reset_index(drop=True)

metrics_df

,Angular range (deg),I0,Method,MSE,PSNR,SSIM
0,180,100000.0,FBP,0.000204,36.91,0.8908
1,180,100000.0,GD,0.000196,37.08,0.9539
2,120,100000.0,FBP,0.006017,22.21,0.5366
3,120,100000.0,GD,0.002949,25.30,0.7101
4,40,100000.0,FBP,0.034739,14.59,0.2953
5,40,100000.0,GD,0.009490,20.23,0.5646
6,180,1000.0,FBP,0.018622,17.30,0.2643
7,180,1000.0,GD,0.001972,27.05,0.5165
8,120,1000.0,FBP,0.024579,16.09,0.2371
9,120,1000.0,GD,0.004507,23.46,0.4474


## 13. Metric plots

In [ ]:
def plot_limited_angle_metrics(
    metrics_df,
    angle_configs,
    sigma,
    I0_colors=None,
    methods=("FBP", "GD"),
    figsize=(15, 7),
    save_path=None,
):
    if I0_colors is None:
        I0_colors = {1e5: "tab:green", 1e3: "tab:orange", 1e2: "tab:red"}

    angle_vals = sorted([a for a, _ in angle_configs])

    fig, axes = plt.subplots(2, 3, figsize=figsize, sharex=True, squeeze=False)

    for row, method in enumerate(methods):
        sub = metrics_df[metrics_df["Method"] == method]

        for I0, color in I0_colors.items():
            s = sub[sub["I0"] == I0].sort_values("Angular range (deg)")
            angles = s["Angular range (deg)"].values
            kw = dict(color=color, marker="o", linewidth=2, markersize=6,
                      label=f"I0 = {I0:.0e}")
            axes[row, 0].semilogy(angles, s["MSE"].values, **kw)
            axes[row, 1].plot(angles, s["PSNR"].values, **kw)
            axes[row, 2].plot(angles, s["SSIM"].values, **kw)

        for col, (ylabel, title) in enumerate([
            ("MSE (log scale)", f"{method} — MSE"),
            ("PSNR (dB)",       f"{method} — PSNR"),
            ("SSIM",            f"{method} — SSIM"),
        ]):
            ax = axes[row, col]
            ax.set_xticks(angle_vals)
            ax.set_xticklabels([f"{a}°" for a in angle_vals])
            ax.set_xlabel("Angular range")
            ax.set_ylabel(ylabel)
            ax.set_title(title, fontweight="semibold")
            ax.grid(True, which="both", alpha=0.3)
            ax.legend(fontsize=8)

    plt.suptitle(
        f"Limited angle — Poisson + Gaussian noise (sigma={sigma}) — metrics",
        fontsize=13, fontweight="semibold",
    )
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


In [14]:
plot_limited_angle_metrics(
    metrics_df=metrics_df,
    angle_configs=angle_configs,
    sigma=sigma,
    save_path=f"{SAVE_DIR}/metrics_vs_angle.png",
)


/var/folders/2f/7345nh9n0nlcj7ty8bmywp480000gn/T/ipykernel_28956/818770639.py:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 14. Discussion — Limited angle artefacts

### What goes wrong with a limited angular range?

By the **Fourier Slice Theorem**, each projection at angle $\theta$ fills one radial
line through the origin of the 2D Fourier transform of the image. When only a
limited range of angles is measured, a wedge-shaped region of Fourier space is
never sampled — the *missing wedge*.

- **180deg (full range)**: all radial directions sampled; FBP and GD recover the
  image well (noise limited by I0 and sigma).
- **120deg**: one-third of Fourier space is missing. Edges perpendicular to the
  missing directions become blurred or streaked.
- **40deg**: the majority of Fourier space is unsampled. Only structures aligned
  with the narrow measured wedge are well recovered; severe elongation artefacts
  appear along the missing direction.

### What is the mathematical principle? When can you see an object edge?

The governing principle is the **Fourier Slice Theorem**:

$$\mathcal{F}_1\{p_\theta\}(\xi) = \mathcal{F}_2\{f\}(\xi\cos\theta,\, \xi\sin\theta)$$

Each 1D Fourier transform of a projection at angle $\theta$ equals the 2D Fourier
transform of the image evaluated along the radial line at that angle.
A sharp edge in the image generates a ridge of energy in 2D Fourier space
**perpendicular to the edge**. Therefore:

> **An edge is reconstructible if and only if its normal direction lies within the
> measured angular range.**

Concretely, if the scan covers $[0°, \alpha]$, then an edge whose outward normal
points at angle $\phi$ can be recovered only when $\phi \in [0°, \alpha]$.
Edges whose normals fall in the missing wedge $(\alpha, 180°)$ are invisible to
the reconstruction — they appear blurred or entirely absent.

This explains why limited-angle CT causes **directional blurring**: horizontal edges
(whose normals point vertically, outside the scan arc for small $\alpha$) are lost,
while edges parallel to the scan arc are preserved. The artefact manifests as
elongation of structures in the direction of the missing wedge.

### Why does GD outperform FBP for limited angles?

FBP applies the ramp filter and backprojects in one pass; the ramp amplifies
high-frequency components and has no mechanism to handle the missing wedge
other than implicitly treating it as zero.

Gradient descent iteratively minimises $\|Ax - b\|^2$ using only the measured
projections. It does not fill in missing Fourier content, but it also does not
amplify the missing-wedge gap the way the ramp filter does, leading to lower
streak artefacts at the cost of some blurring.

Adding a **TV or L1 regulariser** (Practical 2, Q8) to the gradient descent
objective can further suppress missing-wedge artefacts by promoting
piecewise-constant solutions consistent with the measured data.

### Effect of dose (I0) in limited angle

The interaction between limited angle and low dose is additive: each degrades
the reconstruction independently. At 40deg and I0=1e2, the reconstruction is
dominated by both missing-wedge elongation and photon noise simultaneously.
GD handles this combination better than FBP, but neither method produces
diagnostically useful images without regularisation.

### Clinical implication

Digital Breast Tomosynthesis (DBT) acquires projections over ~15–50deg at doses
comparable to 2D mammography. The limited-angle artefact (blurring perpendicular
to the scan arc) is one of the primary limitations of DBT, causing some lesions
to be obscured. Current clinical DBT systems use iterative reconstruction with
TV-like regularisation to mitigate this — directly analogous to what GD+TV
would offer over plain FBP in these experiments.